# Day 08 - Benchmarking and autotuning CuTe kernels

This lesson follows official
[Notebook 10](../notebooks/10_benchmark_autotune.ipynb) and reuses
the predicated elementwise mapping from Day 04. The new problem is
experimental method:

```text
prove correctness -> define candidates -> tune fairly
                  -> compile the winner -> benchmark steadily
```

You will compare 64-bit and 128-bit copy widths, use both automatic
and explicit tuning APIs, and convert elapsed microseconds into
effective memory bandwidth.

In [ ]:
import torch

import cutlass
import cutlass.cute as cute
import cutlass.testing as testing
from cutlass.cute.testing import autotune_jit

cutlass.cuda.initialize_cuda_context()

## 1. Keep four activities separate

| Activity | Question | Should be inside timing? |
| --- | --- | --- |
| Correctness | Does every candidate compute `C = A + B`? | No |
| JIT compilation | Can CuTe specialize this Python program? | No |
| Tuning | Which valid configuration is fastest here? | Measures candidates |
| Final benchmark | How fast is the selected operation? | Kernel invocation only |

Timing a first call commonly measures Python tracing, MLIR
lowering, module loading, allocation, and GPU work together. That
number does not describe kernel performance. Compile first, warm
up, synchronize through the timing utility, and then measure.

## 2. A compile-time parameter changes the generated kernel

`copy_bits` is a `cutlass.Constexpr`, so each candidate becomes a
separate specialization:

```python
vector_size = copy_bits // tensor_a.element_type.width
```

For Float32, 64-bit and 128-bit copies correspond to 2 and 4
elements. That value changes the value layout and therefore the CTA
tile width. Tuning is choosing between generated programs, not
changing a runtime branch.

In [ ]:
dtype_width = 32
thread_shape = (4, 32)

for copy_bits in (64, 128):
    vector_size = copy_bits // dtype_width
    value_shape = (4, vector_size)
    tile_shape = (
        thread_shape[0] * value_shape[0],
        thread_shape[1] * value_shape[1],
    )
    print(
        f"{copy_bits:3} bits -> {vector_size} Float32 values/copy "
        f"-> CTA tile {tile_shape}"
    )

## 3. Decorator-based autotuning

`@autotune_jit` wraps a `@cute.jit` function and searches a
parameter dictionary:

```python
@autotune_jit(
    params_dict={"copy_bits": [64, 128]},
    update_on_change=["m", "n"],
    warmup_iterations=5,
    iterations=20,
)
@cute.jit
def add(..., copy_bits: cutlass.Constexpr = 128):
    ...
```

`update_on_change` defines the cache key dimensions that require a
new search. A best choice for one `(m, n)` may not remain best for
another shape. Include every runtime property that can change the
ranking; avoid including unrelated values that cause needless
retuning.

Set `CUTE_DSL_LOG_AUTOTUNE=1` before execution when you need to see
candidate times and cache hits.

## 4. Explicit `testing.tune`

The explicit API makes the winning dictionary visible. Its callback
has an important two-level contract:

```python
def compile_config(a, b, c, rows, cols, copy_bits):
    compiled = cute.compile(
        add, a, b, c, rows, cols, copy_bits=copy_bits
    )
    return lambda: compiled(a, b, c, rows, cols)
```

The outer callback receives one candidate and compiles it. The
returned zero-argument callable performs only the runtime launch.
Returning the result of the launch, instead of a callable, would
execute too early and leave the tuner nothing repeatable to time.

## 5. Reliable benchmark inputs

`testing.benchmark` returns average time in **microseconds**. Its
controls address different sources of noise:

- `warmup_iterations`: populate code/data paths and stabilize clocks;
- `iterations`: amortize timer noise;
- `stream`: use the same stream as the measured launch;
- `workspace_generator`: construct independent argument sets;
- `workspace_count`: rotate those sets to reduce persistent L2 hits;
- `use_cuda_graphs`: reduce host launch overhead for very short work.

Workspace rotation changes what is being measured. One workspace
can describe warm-cache reuse; several larger workspaces better
approximate streaming traffic. State which experiment you chose.

In [ ]:
def effective_bandwidth_gbps(m, n, element_bytes, average_us):
    # Two input reads plus one output write.
    bytes_moved = 3 * m * n * element_bytes
    return bytes_moved / (average_us * 1_000)


example = effective_bandwidth_gbps(
    m=4096, n=4096, element_bytes=4, average_us=200
)
print(f"Example effective bandwidth: {example:.2f} GB/s")

## 6. A fair tuning checklist

Before comparing candidates, verify:

1. Every candidate passes correctness, including edge shapes.
2. Candidate compilation is outside the measured callback.
3. All candidates use identical inputs, stream, warmup, and trials.
4. The search space contains only legal alignments and vector widths.
5. The winner is recompiled or retrieved and checked again.
6. The final benchmark reports units and the byte/FLOP convention.

A tuner can faithfully select the fastest incorrect kernel. Tuning
never substitutes for validation.

## 7. Challenge: complete both tuning paths

Fill six TODOs:

1. Add the `@autotune_jit` search over 64 and 128 bits.
2. Convert copy width into elements per vector.
3. Launch one CTA for each outer tile.
4. Compile one explicit-tuning candidate.
5. Return the winner from `testing.tune`.
6. Benchmark and report microseconds.

The data mapping is already complete so you can focus on experiment
construction.

In [ ]:
@cute.kernel
def add_kernel(
    g_a: cute.Tensor,
    g_b: cute.Tensor,
    g_c: cute.Tensor,
    coordinates: cute.Tensor,
    shape: cute.Shape,
    thread_layout: cute.Layout,
    value_layout: cute.Layout,
):
    thread_idx, _, _ = cute.arch.thread_idx()
    block_idx, _, _ = cute.arch.block_idx()
    block_coord = ((None, None), block_idx)
    atom = cute.make_copy_atom(cute.nvgpu.CopyUniversalOp(), g_a.element_type)
    tiled_copy = cute.make_tiled_copy_tv(atom, thread_layout, value_layout)
    thread_copy = tiled_copy.get_slice(thread_idx)
    a = thread_copy.partition_S(g_a[block_coord])
    b = thread_copy.partition_S(g_b[block_coord])
    c = thread_copy.partition_D(g_c[block_coord])
    coord = thread_copy.partition_S(coordinates[block_coord])
    fragment_a = cute.make_fragment_like(a)
    fragment_b = cute.make_fragment_like(b)
    fragment_c = cute.make_fragment_like(c)
    pred = cute.make_rmem_tensor(coord.shape, cutlass.Boolean)
    for i in cutlass.range_constexpr(cute.size(pred)):
        pred[i] = cute.elem_less(coord[i], shape)
    cute.copy(atom, a, fragment_a, pred=pred)
    cute.copy(atom, b, fragment_b, pred=pred)
    fragment_c.store(fragment_a.load() + fragment_b.load())
    cute.copy(atom, fragment_c, c, pred=pred)


# TODO(1): add @autotune_jit with copy_bits=[64, 128],
# update_on_change=["m", "n"], and short warmup/measurement counts.
@cute.jit
def add(tensor_a, tensor_b, tensor_c, m, n, copy_bits: cutlass.Constexpr = 128):
    # TODO(2): convert copy_bits to an element count using dtype.width.
    raise NotImplementedError("Day 08 TODO(2): compute vector_size")

    thread_layout = cute.make_ordered_layout((4, 32), order=(1, 0))
    value_layout = cute.make_ordered_layout((4, vector_size), order=(1, 0))
    tile_shape, tv_layout = cute.make_layout_tv(thread_layout, value_layout)
    g_a = cute.zipped_divide(tensor_a, tile_shape)
    g_b = cute.zipped_divide(tensor_b, tile_shape)
    g_c = cute.zipped_divide(tensor_c, tile_shape)
    coordinates = cute.zipped_divide(
        cute.make_identity_tensor(tensor_c.shape), tile_shape
    )

    # TODO(3): launch add_kernel with one CTA per outer g_c tile and one
    # thread per TV-layout thread coordinate.


def tune_add(tensor_a, tensor_b, tensor_c, m: int, n: int):
    # TODO(4): define a callback that compiles one copy_bits configuration and
    # returns a zero-argument lambda that invokes the compiled function.
    # TODO(5): return testing.tune(...), passing params_dict and JitArguments.
    raise NotImplementedError("Day 08 TODO(4-5): implement explicit tuning")


def run(m: int, n: int) -> None:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("Day 08 needs a CUDA device.")
    cutlass.cuda.initialize_cuda_context()
    a = torch.randn(m, n, device="cuda", dtype=torch.float32)
    b = torch.randn(m, n, device="cuda", dtype=torch.float32)
    c = torch.empty_like(a)
    best = tune_add(a, b, c, m, n)
    compiled = cute.compile(add, a, b, c, m, n, **best)
    compiled(a, b, c, m, n)
    torch.cuda.synchronize()
    torch.testing.assert_close(c, a + b)

    # TODO(6): benchmark `add` with testing.benchmark and print microseconds.
    print(f"best configuration={best}")

## 8. Run correctness, tuning, and measurement

Run only after all TODOs are filled. Repeat with a larger,
non-square shape and observe whether the winning copy width changes.

In [ ]:
run(512, 512)
run(1024, 1536)

## 9. Interpret the result rather than only printing it

Record:

| Shape | Winning copy width | Average us | Effective GB/s |
| --- | ---: | ---: | ---: |
| `512 x 512` | fill in | fill in | fill in |
| `1024 x 1536` | fill in | fill in | fill in |

Then answer:

- Did the wider copy always win?
- Is the small shape dominated by launch overhead?
- Does rotating workspaces lower apparent bandwidth?
- Would `use_cuda_graphs=True` change the kernel or only submission?

Day 09 isolates the last question by capturing repeated launches.